# Лабораторная работа №3 — решающее дерево

## 1. Выбор начальных условий

- **Spambase:** классифицируем письма как спам или не спам по 57 числовым признакам. F1-macro одинаково учитывает оба класса; accuracy показывает долю верных ответов.
- **SML2010:** прогнозируем температуру в столовой на 15 минут вперёд по 18 текущим признакам. MAE показывает типичную величину ошибки в °C, RMSE сильнее реагирует на крупные промахи.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from data_setup import DINING_TEMPERATURE, RANDOM_STATE, load_sml2010, load_spambase, sml2010_cv, spambase_cv

Xc_train, Xc_test, yc_train, yc_test = load_spambase()
Xr_train, Xr_test, yr_train, yr_test = load_sml2010()
print(f"Spambase: {Xc_train.shape} / {Xc_test.shape}")
print(f"SML2010:  {Xr_train.shape} / {Xr_test.shape}")

Spambase: (3363, 57) / (841, 57)
SML2010:  (2763, 18) / (1370, 18)


## 2. Создание бейзлайна и оценка качества

Обучаем `DecisionTreeClassifier` с критерием Джини и `DecisionTreeRegressor` с `squared_error`, без ограничения глубины и с минимальным листом в один объект. Масштабирование не требуется: разбиения опираются на порядок значений. Фиксируем `random_state` в sklearn и оцениваем обе модели на тесте; для температуры сохраняем ориентир «прогноз равен текущему значению».

In [2]:
def classification_metrics(y, prediction):
    return {"f1_macro": f1_score(y, prediction, average="macro"), "accuracy": accuracy_score(y, prediction)}


def regression_metrics(y, prediction):
    return {"mae": mean_absolute_error(y, prediction), "rmse": root_mean_squared_error(y, prediction)}


clf = DecisionTreeClassifier(random_state=RANDOM_STATE)
reg = DecisionTreeRegressor(random_state=RANDOM_STATE)
clf.fit(Xc_train, yc_train)
reg.fit(Xr_train, yr_train)
classification_scores = {"sklearn_base": classification_metrics(yc_test, clf.predict(Xc_test))}
regression_scores = {"sklearn_base": regression_metrics(yr_test, reg.predict(Xr_test))}
persistence = regression_metrics(yr_test, Xr_test[DINING_TEMPERATURE])

display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict({**regression_scores, "текущая температура": persistence}, orient="index"))

,f1_macro,accuracy
sklearn_base,0.919255,0.922711


,mae,rmse
sklearn_base,0.331432,0.722967
текущая температура,0.116360,0.133181


## 3. Улучшение бейзлайна

### Гипотезы

1. Ограничение глубины может уменьшить переобучение по сравнению с деревом без ограничения.
2. Увеличение минимального размера листа может сделать предсказания устойчивее: один лист не строится по нескольким случайным объектам.

### Проверка гипотез

Перебираем глубину `None, 4, 8, 12` и минимальный размер листа `1, 5, 15` на обучающих данных. Для классификации — три стратифицированных фолда и F1-macro, для регрессии — три временных фолда с зазором и MAE. Показываем результаты всех комбинаций, включая исходную.

In [3]:
grid = {"max_depth": [None, 4, 8, 12], "min_samples_leaf": [1, 5, 15]}
clf_search = GridSearchCV(clf, grid, scoring="f1_macro", cv=spambase_cv()).fit(Xc_train, yc_train)
reg_search = GridSearchCV(reg, grid, scoring="neg_mean_absolute_error", cv=sml2010_cv()).fit(Xr_train, yr_train)


def cv_table(search, metric, sign=1):
    results = search.cv_results_
    return pd.DataFrame({
        "max_depth": ["без ограничения" if d is None else str(d) for d in results["param_max_depth"]],
        "min_samples_leaf": results["param_min_samples_leaf"],
        metric: sign * results["mean_test_score"],
    }).sort_values(metric, ascending=(sign == -1))


display(cv_table(clf_search, "CV F1-macro"))
display(cv_table(reg_search, "CV MAE, °C", sign=-1))

,max_depth,min_samples_leaf,CV F1-macro
9,12,1,0.904193
1,без ограничения,5,0.902427
10,12,5,0.902334
0,без ограничения,1,0.900120
6,8,1,0.898821
7,8,5,0.895237
3,4,1,0.892186
4,4,5,0.891479
2,без ограничения,15,0.882324
11,12,15,0.880944


,max_depth,min_samples_leaf,"CV MAE, °C"
9,12,1,0.152595
6,8,1,0.153975
0,без ограничения,1,0.155279
1,без ограничения,5,0.162917
10,12,5,0.162917
7,8,5,0.165910
2,без ограничения,15,0.194860
8,8,15,0.194860
11,12,15,0.194860
3,4,1,0.302019


### Улучшенный бейзлайн

`GridSearchCV` выбирает настройки по средним метрикам на фолдах и переобучает выбранную модель на всей обучающей выборке. Сравниваем её с базовым деревом на неизменном тесте.

In [4]:
print("Spambase:", clf_search.best_params_)
print("SML2010:", reg_search.best_params_)
classification_scores["sklearn_improved"] = classification_metrics(yc_test, clf_search.predict(Xc_test))
regression_scores["sklearn_improved"] = regression_metrics(yr_test, reg_search.predict(Xr_test))
display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict(regression_scores, orient="index"))

Spambase: {'max_depth': 12, 'min_samples_leaf': 1}
SML2010: {'max_depth': 12, 'min_samples_leaf': 1}


,f1_macro,accuracy
sklearn_base,0.919255,0.922711
sklearn_improved,0.927685,0.931034


,mae,rmse
sklearn_base,0.331432,0.722967
sklearn_improved,0.316183,0.695282


**Выводы.** Для обеих задач кросс-валидация выбрала глубину `12` и `min_samples_leaf=1`. Увеличение минимального размера листа не дало лучшего результата. На тесте F1-macro вырос с 0,9193 до 0,9277, а MAE снизилась с 0,3314 до 0,3162 °C. В регрессии простой прогноз по текущей температуре всё равно точнее (MAE 0,1164 °C).

## 4. Имплементация алгоритма машинного обучения

### Базовая реализация

Используем один класс для бинарной классификации и регрессии. В каждом узле сортируем значения каждого признака и проверяем пороги между соседними различными значениями. Накопленные суммы и суммы квадратов позволяют вычислить стоимость всех порогов без повторного построения подвыборок. Лист хранит среднее целевых значений: для бинарной цели это доля класса `1`.

Стоимость ветви — сумма квадратов отклонений `SSE = Σy² − (Σy)²/n`. Для цели `0/1` взвешенный критерий Джини равен `2 × SSE`, для регрессии взвешенная квадратичная ошибка равна `SSE`. Центрирование цели в узле не меняет SSE и уменьшает численную погрешность. Выбираем разбиение с наибольшим снижением суммы стоимостей. Если снизить её не удалось, глубина исчерпана или ветви вышли бы слишком малы, оставляем лист.

In [5]:
class DecisionTreeNumPy:
    def __init__(self, classification=True, max_depth=None, min_samples_leaf=1):
        self.classification = classification
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf

    def fit(self, X, y):
        self.tree = self._grow(np.asarray(X, dtype=float), np.asarray(y, dtype=float), depth=0)
        return self

    def _grow(self, X, y, depth):
        n = len(y)
        leaf = float(y.mean())
        minimum = self.min_samples_leaf
        if n < 2 * minimum or (self.max_depth is not None and depth >= self.max_depth):
            return leaf

        centered = y - leaf
        factor = 2 if self.classification else 1
        best_loss = factor * np.dot(centered, centered)
        if best_loss <= 1e-12:
            return leaf

        split = None
        for feature in range(X.shape[1]):
            order = np.argsort(X[:, feature], kind="stable")
            values = X[order, feature]
            candidates = np.flatnonzero(values[:-1] < values[1:]) + 1
            counts = candidates[(candidates >= minimum) & (n - candidates >= minimum)]
            if not len(counts):
                continue

            targets = centered[order]
            sums = np.cumsum(targets)
            squares = np.cumsum(targets ** 2)
            left_sum, left_sq = sums[counts - 1], squares[counts - 1]
            right_sum, right_sq = sums[-1] - left_sum, squares[-1] - left_sq
            left_loss = left_sq - left_sum ** 2 / counts
            right_loss = right_sq - right_sum ** 2 / (n - counts)
            losses = factor * (left_loss + right_loss)
            i = np.argmin(losses)
            if losses[i] < best_loss - 1e-12:
                best_loss = losses[i]
                pos = counts[i]
                split = (feature, (values[pos - 1] + values[pos]) / 2)

        if split is None:
            return leaf

        feature, threshold = split
        left = X[:, feature] <= threshold
        return (feature, threshold, self._grow(X[left], y[left], depth + 1), self._grow(X[~left], y[~left], depth + 1))

    def predict(self, X):
        predictions = []
        for row in np.asarray(X, dtype=float):
            node = self.tree
            while isinstance(node, tuple):
                feature, threshold, left, right = node
                node = left if row[feature] <= threshold else right
            predictions.append(node)
        result = np.asarray(predictions)
        return (result > 0.5).astype(int) if self.classification else result

Проверяем порог на простой классификации и среднее в листе на регрессии. Отдельно проверяем, что слишком большой `min_samples_leaf` оставляет один лист. Затем обучаем обе собственные модели с базовыми параметрами и сравниваем с пунктом 2.

In [6]:
X_toy = [[0], [1], [2], [3]]
np.testing.assert_array_equal(DecisionTreeNumPy().fit(X_toy, [0, 0, 1, 1]).predict(X_toy), [0, 0, 1, 1])
toy_reg = DecisionTreeNumPy(classification=False, max_depth=1).fit(X_toy, [0, 2, 10, 12])
np.testing.assert_allclose(toy_reg.predict(X_toy), [1, 1, 11, 11])
toy_leaf = DecisionTreeNumPy(classification=False, min_samples_leaf=3).fit(X_toy, [0, 2, 10, 12])
np.testing.assert_allclose(toy_leaf.predict(X_toy), [6, 6, 6, 6])

clf_own = DecisionTreeNumPy().fit(Xc_train, yc_train)
reg_own = DecisionTreeNumPy(classification=False).fit(Xr_train, yr_train)
classification_scores["custom_base"] = classification_metrics(yc_test, clf_own.predict(Xc_test))
regression_scores["custom_base"] = regression_metrics(yr_test, reg_own.predict(Xr_test))
base_variants = ["sklearn_base", "custom_base"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[base_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[base_variants])

,f1_macro,accuracy
sklearn_base,0.919255,0.922711
custom_base,0.914373,0.917955


,mae,rmse
sklearn_base,0.331432,0.722967
custom_base,0.323886,0.716787


**Сравнение с пунктом 2.** Базовый NumPy-классификатор набрал F1-macro 0,9144 против 0,9193 у sklearn. Базовое дерево NumPy для регрессии дало MAE 0,3239 °C против 0,3314 °C. При равных или очень близких стоимостях порогов реализация и sklearn могут выбрать разные разбиения, поэтому совпадение деревьев не требуется.

### 4.2. Улучшенная реализация

Переносим `max_depth` и `min_samples_leaf` из пункта 3 без повторного подбора. Обучаем собственные деревья и сравниваем с улучшенными деревьями sklearn на том же тесте.

In [7]:
clf_own_improved = DecisionTreeNumPy(**clf_search.best_params_).fit(Xc_train, yc_train)
reg_own_improved = DecisionTreeNumPy(classification=False, **reg_search.best_params_).fit(Xr_train, yr_train)
classification_scores["custom_improved"] = classification_metrics(yc_test, clf_own_improved.predict(Xc_test))
regression_scores["custom_improved"] = regression_metrics(yr_test, reg_own_improved.predict(Xr_test))
improved_variants = ["sklearn_improved", "custom_improved"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[improved_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[improved_variants])

,f1_macro,accuracy
sklearn_improved,0.927685,0.931034
custom_improved,0.926398,0.929845


,mae,rmse
sklearn_improved,0.316183,0.695282
custom_improved,0.323563,0.716681


**Сравнение с пунктом 3.** При переносе глубины `12` F1-macro собственного дерева вырос до 0,9264 (у sklearn 0,9277). В регрессии ограничение глубины почти не изменило NumPy MAE: 0,3236 °C против 0,3239 °C у базового дерева NumPy и 0,3162 °C у улучшенного sklearn.

In [8]:
variants = ["sklearn_base", "sklearn_improved", "custom_base", "custom_improved"]
rows = []
for task, scores in [("classification", classification_scores), ("regression", regression_scores)]:
    for variant in variants:
        rows.append({"lab": 3, "task": task, "algorithm": "DecisionTree", "variant": variant, **scores[variant]})

results = pd.DataFrame(rows)
results.to_csv(root / "results" / "lab03.csv", index=False, float_format="%.8f")
for task in results["task"].unique():
    display(results.loc[results["task"] == task].dropna(axis=1, how="all").reset_index(drop=True))

,lab,task,algorithm,variant,f1_macro,accuracy
0,3,classification,DecisionTree,sklearn_base,0.919255,0.922711
1,3,classification,DecisionTree,sklearn_improved,0.927685,0.931034
2,3,classification,DecisionTree,custom_base,0.914373,0.917955
3,3,classification,DecisionTree,custom_improved,0.926398,0.929845


,lab,task,algorithm,variant,mae,rmse
0,3,regression,DecisionTree,sklearn_base,0.331432,0.722967
1,3,regression,DecisionTree,sklearn_improved,0.316183,0.695282
2,3,regression,DecisionTree,custom_base,0.323886,0.716787
3,3,regression,DecisionTree,custom_improved,0.323563,0.716681
